# Data Preparation

In [34]:
# import libraries
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

## 1. Data Understanding, Cleaning & Transformation

In [35]:
# Ingest data
fuel_mix_elec_gen = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/AnnualFuelMixforElectricityGenerationbyEnergyProducts2005toJun2021.csv')
elec_gen_and_consump = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/ElectricityGenerationAndConsumptionAnnual.csv')
ggas_emission_gas_type = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/GreenhouseGasEmissionsByGasTypeAnnual.csv')
ggas_emission_sector = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/GreenhouseGasEmissionsBySectorAnnual.csv')
per_capita_gdp_chained = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/PerCapitaGDPInChained2015DollarsAnnual.csv')
renewable_energy_share = pd.read_csv('C:/Users/Admin/Desktop/net-zero emission by 2025 project/net-zero-emission-data-analytics-project/data sources/data.gov.sg/RenewableEnergyShareInTheTotalFinalEnergyConsumptionAnnual.csv')

In [36]:
# See number of records and columns in each dataset
datasets = {
    "Annual Fuel Mix for Electricity Generation by Energy Products 2005 to Jun 2021": fuel_mix_elec_gen,
    "Electricity Generation and Consumption Annual": elec_gen_and_consump,
    "Greenhouse Gas Emissions by Gas Type Annual": ggas_emission_gas_type,
    "Greenhouse Gas Emissions by Sector Annual": ggas_emission_sector,
    "Per Capita GDP in Chained 2015 Dollars Annual": per_capita_gdp_chained,
    "Renewable Energy Share in Total Final Energy Consumption Annual": renewable_energy_share,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

Annual Fuel Mix for Electricity Generation by Energy Products 2005 to Jun 2021: 68 rows, 3 columns
Electricity Generation and Consumption Annual: 18 rows, 52 columns
Greenhouse Gas Emissions by Gas Type Annual: 15 rows, 25 columns
Greenhouse Gas Emissions by Sector Annual: 13 rows, 5 columns
Per Capita GDP in Chained 2015 Dollars Annual: 8 rows, 67 columns
Renewable Energy Share in Total Final Energy Consumption Annual: 1 rows, 16 columns


From the above output, we can see that datasets are small, with per capita GDP dataset having the most records of 67. Next, we will look at each dataset to understand more about their data.

In [37]:
# Look at Annual Fuel Mix for Electricity Generation by Energy Products 2005 to Jun 2021 dataset
fuel_mix_elec_gen.head(5)

,year,energy_products,percentage
0,2005,Petroleum Products,23.1
1,2005,Natural Gas,74.4
2,2005,Coal,0.0
3,2005,Others,2.5
4,2006,Petroleum Products,19.8


The output above shows that the dataset has *year*, *energy_products*, and *percentage* columns, showing contribution of electricity generation in % each year for each energy product. We will not transform this dataset as the data is already easily understandable and with proper columns. 

Before moving on to look at next dataset, we will **check whether there are any missing or duplicate records** in this dataset.

In [42]:
# Check missing values
fuel_mix_elec_gen.isna().sum()


year               0
energy_products    0
percentage         0
dtype: int64

In [43]:
# Check duplicate values
fuel_mix_elec_gen.duplicated().sum()

0

From the above cells, we can see that there are no missing or duplicate records present in the dataset. We will also make sure data types are correct to ensure high data quality.

In [72]:
# Check data types
fuel_mix_elec_gen.dtypes

year                 int64
energy_products     object
percentage         float64
dtype: object

The output above shows the correct data types. Lastly, for this dataset, we will make sure text column (*energy_products*) is clean and make sure no truncation is needed by checking any leading or trailing whitespaces in the values. 

In [81]:
# Check for leading and trailing spaces in energy_products column
fuel_mix_elec_gen['energy_products'].unique()

array(['Petroleum Products', 'Natural Gas', 'Coal', 'Others'],
      dtype=object)

We can see from the above output that the values have no leading or trailing whitespaces, therefore, there is no need for truncation. Next, let's take a look at Electricity Generation and Consumption Annual dataset.

In [22]:
# Look at Electricity Generation and Consumption Annual dataset
elec_gen_and_consump.head()

,DataSeries,2025,2024,2023,2022,2021,2020,2019,2018,2017,...,1984,1983,1982,1981,1980,1979,1978,1977,1976,1975
0,Electricity Generation,60410.4,59616.1,57384.2,57113.7,55788.3,53071.6,54142.3,52904.8,52225.8,...,9420.7,8625.9,7859.5,7441.9,6940.5,6447.8,5897.9,5114.7,4604.9,4175.7
1,Electricity Consumption,na,57639.7,55426.4,54910.6,53495.6,50789.7,51736.7,50466.9,49643.7,...,na,na,na,na,na,na,na,na,na,na
2,Industrial-Related,na,22704.0,22103.0,22693.9,22293.1,20978.9,21463.9,21441.8,21240.6,...,na,na,na,na,na,na,na,na,na,na
3,Manufacturing,na,20630.5,20171.2,20682.1,20284.8,19121.3,19439.4,19441.0,19306.8,...,na,na,na,na,na,na,na,na,na,na
4,Construction,na,440.0,421.6,464.5,472.0,387.0,397.1,431.5,485.5,...,na,na,na,na,na,na,na,na,na,na


As we can see from the output, this dataset has DataSeries column, which shows electricity consumption by sectors, along with total electricity generation and consumption value, in Gigawatt Hours, with the rest of the columns being year columns from 1975 to 2025. Therefore, this needs to be transformed, into one year column.

We will also add a category column with the value 'Total' (for Electricity Generation and Electricity Consumption rows in DataSeries column) or 'Consumption Sector' (for the rest of the rows in DataSeries column). Therefore, the final transformed dataset will have:
1. Year
2. Category
3. DataSeries
4. Value (in Gigawatt Hours)

Before, transforming, we will quickly check for **missing or duplicate values** in the dataset. If there are none, we will proceed to transform the dataset as discussed.

In [ ]:
# Check missing values
elec_gen_and_consump.isna().sum()

DataSeries    0
2025          0
2024          0
2023          0
2022          0
2021          0
2020          0
2019          0
2018          0
2017          0
2016          0
2015          0
2014          0
2013          0
2012          0
2011          0
2010          0
2009          0
2008          0
2007          0
2006          0
2005          0
2004          0
2003          0
2002          0
2001          0
2000          0
1999          0
1998          0
1997          0
1996          0
1995          0
1994          0
1993          0
1992          0
1991          0
1990          0
1989          0
1988          0
1987          0
1986          0
1985          0
1984          0
1983          0
1982          0
1981          0
1980          0
1979          0
1978          0
1977          0
1976          0
1975          0
dtype: int64

In [ ]:
# Check duplicate values
elec_gen_and_consump.duplicated().sum()

0

Because there are no missing or duplicate records, we can now proceed to do the transformation.

In [51]:
# Transform Electricity Generation and Consumption Annual dataset
year_cols = [col for col in elec_gen_and_consump.columns if col != 'DataSeries']

elec_gen_and_consump_transformed = elec_gen_and_consump.melt(
    id_vars=['DataSeries'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Value'
)

elec_gen_and_consump_transformed['Category'] = np.where(
    elec_gen_and_consump_transformed['DataSeries'].isin([
        'Electricity Generation',
        'Electricity Consumption'
    ]),
    'Total',
    'Consumption Sector'
)

# Convert Year to integer
elec_gen_and_consump_transformed['Year'] = (
    elec_gen_and_consump_transformed['Year']
    .astype(int)
)

# Convert Value to numeric and handle "na"
elec_gen_and_consump_transformed['Value'] = pd.to_numeric(
    elec_gen_and_consump_transformed['Value'],
    errors='coerce'
)

# Reorder columns
elec_gen_and_consump_transformed = elec_gen_and_consump_transformed[
    ['Year', 'Category', 'DataSeries', 'Value']
]

elec_gen_and_consump_transformed.head()

,Year,Category,DataSeries,Value
0,2025,Total,Electricity Generation,60410.4
1,2025,Total,Electricity Consumption,NaN
2,2025,Consumption Sector,Industrial-Related,NaN
3,2025,Consumption Sector,Manufacturing,NaN
4,2025,Consumption Sector,Construction,NaN


As we can see above, the transformation looks correct, giving *Year*, *Category*, *DataSeries*, and *Value* columns as we expected. We will also check data types of each column to maintain data quality for later analysis.

In [52]:
# Check data types
elec_gen_and_consump_transformed.dtypes

Year            int32
Category       object
DataSeries     object
Value         float64
dtype: object

We found out that data types are correct. Next, we will ensure there are no missing or duplicate values after transformation. First, we will check missing values.

In [61]:
# Check missing values
elec_gen_and_consump_transformed.isna().sum()

Year            0
Category        0
DataSeries      0
Value         527
dtype: int64

Interestingly, there are now missing values in *Value* column. We will find out why by checking the records that has missing values in *Value* column.

In [ ]:
# Check which records has missing values in the column "Value"
elec_gen_and_consump_transformed[
    elec_gen_and_consump_transformed['Value'].isna()
]

,Year,Category,DataSeries,Value
1,2025,Total,Electricity Consumption,NaN
2,2025,Consumption Sector,Industrial-Related,NaN
3,2025,Consumption Sector,Manufacturing,NaN
4,2025,Consumption Sector,Construction,NaN
5,2025,Consumption Sector,Utilities,NaN
...,...,...,...,...
913,1975,Consumption Sector,"Professional, Scientific & Technical, ...",NaN
914,1975,Consumption Sector,Other Commerce And Service-Related,NaN
915,1975,Consumption Sector,Transport-Related,NaN
916,1975,Consumption Sector,Households,NaN


The output tells us that some years, such as 1975 and 2025 as we can see have no electricity consumption data. This aligns with the original data having 'na' values in them, except for Electricity Generation. We will confirm our doubt in the next cell by checking which year has how much missing values in the *Value* column.

In [ ]:
# Check how many records has missing values in the column "Value" by Year
elec_gen_and_consump_transformed[
    elec_gen_and_consump_transformed['Value'].isna()
].groupby('Year').size()

Year
1975    17
1976    17
1977    17
1978    17
1979    17
1980    17
1981    17
1982    17
1983    17
1984    17
1985    17
1986    17
1987    17
1988    17
1989    17
1990    17
1991    17
1992    17
1993    17
1994    17
1995    17
1996    17
1997    17
1998    17
1999    17
2000    17
2001    17
2002    17
2003    17
2004    17
2025    17
dtype: int64

In [74]:
# Check number of unique values in the column "DataSeries"
elec_gen_and_consump_transformed['DataSeries'].nunique()

18

The results show that for the years **1975 to 2004** and **2025**, there are 17 missing values for each year. Since the DataSeries column contains 18 unique categories, this indicates that only Electricity Generation has recorded values for those years, while the remaining 17 data series contain missing values (NaN). 

This pattern suggests that electricity generation data is available for these years, whereas electricity consumption and its sectoral breakdown were not reported in the source dataset. Also, since imputing those values would make the data highly inaccurate, we will decide to remove those records with missing values.

In [ ]:
# Remove missing values in the column "Value" since it is not possible to impute the missing values with a reasonable assumption.
elec_gen_and_consump_transformed = elec_gen_and_consump_transformed.dropna(
    subset=['Value']
)

We will now check the missing values again.

In [ ]:
# Check missing values
elec_gen_and_consump_transformed.isna().sum()

Year          0
Category      0
DataSeries    0
Value         0
dtype: int64

The output above confirms that there are no more missing values in our dataset. Lastly, we will check *Category* and *DataSeries* columns to ensure there are no leading or trailing whitespaces.

In [82]:
# Check leading or trailing whitespaces for the column "Category"
elec_gen_and_consump_transformed['Category'].unique()

array(['Total', 'Consumption Sector'], dtype=object)

In [83]:
# Check leading or trailing whitespaces for the column "DataSeries"
elec_gen_and_consump_transformed['DataSeries'].unique()

array(['Electricity Generation', 'Electricity Consumption',
       '    Industrial-Related', '        Manufacturing',
       '        Construction', '        Utilities',
       '        Other Industrial-Related',
       '    Commerce And Service-Related',
       '        Wholesale And Retail Trade',
       '        Accommodation And Food Services',
       '        Information And Communications',
       '        Financial And Insurance Activities',
       '        Real Estate Activities',
       '        Professional, Scientific & Technical, Administration & Support Activities',
       '        Other Commerce And Service-Related',
       '    Transport-Related', '    Households', '    Others'],
      dtype=object)

The outputs show that there are leading whitespaces present in the *DataSeries* column. Therefore, we will truncate it.

In [84]:
# Truncate the column "DataSeries" to remove leading and trailing whitespaces
elec_gen_and_consump_transformed['DataSeries'] = (
    elec_gen_and_consump_transformed['DataSeries'].str.strip()
)

In [86]:
# Check leading or trailing whitespaces for the column "DataSeries"
elec_gen_and_consump_transformed['DataSeries'].unique()

array(['Electricity Generation', 'Electricity Consumption',
       'Industrial-Related', 'Manufacturing', 'Construction', 'Utilities',
       'Other Industrial-Related', 'Commerce And Service-Related',
       'Wholesale And Retail Trade', 'Accommodation And Food Services',
       'Information And Communications',
       'Financial And Insurance Activities', 'Real Estate Activities',
       'Professional, Scientific & Technical, Administration & Support Activities',
       'Other Commerce And Service-Related', 'Transport-Related',
       'Households', 'Others'], dtype=object)

The above output shows there are no more extra whitespaces in the *DataSeries* column values. One last thing we will do for this dataset is standardize column names to snake_case.



In [91]:
# Standardize the column names to snake case to have consistent naming conventions
elec_gen_and_consump_transformed = elec_gen_and_consump_transformed.rename(
    columns={
        'Year': 'year',
        'Category': 'category',
        'DataSeries': 'data_series',
        'Value': 'value'
    }
)

Now, we will move on to our next dataset, Greenhouse Gas Emissions by Gas Type Annual.